# Descriptors Computation
This notebook handles the descriptors computation. 
This procedure involves the following steps:
- For each variable, we estimate its Markov Blanket (MB) by selecting its lagged versions from one time step before and one after. 
- We standardize the time series to avoid varsortability
- Using the estimated MB, we compute a set of descriptors for all possible causal pairs (i.e., $ t-\tau \rightarrow t, \forall \tau$). that characterize the causal relationship between the variable pairs. These descriptors include conditional mutual information terms and other statistical properties that provide insights into the dependencies and interactions between the variables.
- For families of descriptors, we compute the quantiles of their empirical distributions. This step captures the distributional characteristics and aids in feature representation for the classifier.
- The computed descriptors and their quantiles are compiled into an input feature vector. This vector encapsulates the essential characteristics of the causal relationships and serves as the input for the classifier.
- For training data, each input vector is labeled as causal (1) or noncausal (0) based on the original selection criteria from the synthetic data's Directed Acyclic Graph (DAG). This labeling is crucial for supervised learning and model training.
- The labeled dataset, comprising the feature vectors, is used to train a classifier. The classifier learns to predict the likelihood of causal relationships based on the descriptors.
- For unseen time series data, the trained classifier predicts the probability of causal links for each pair of variables. The predictions are based on the computed descriptors for the test data.

In [ ]:
from d2c.descriptors import D2C, DataLoader

In [ ]:
N_VARS = 5
MAXLAGS = 3
N_JOBS = 50

The DataLoader class for handling time series data and directed acyclic graphs (DAGs) to prepare for the following stage: the descriptors computation.
The preparation includes:
1. Creating lagged time series: This step involves generating lagged versions of the original time series data. Lagged time series help in capturing temporal dependencies and interactions between variables at different time steps.
2. Flattening the original dictionaries into coherent lists: The original data, which may be stored in nested dictionaries, is flattened into lists. This transformation ensures that the data is in a consistent and accessible format for further processing.
3. Renaming the nodes of the DAGs: The nodes of the Directed Acyclic Graphs (DAGs) are renamed to maintain consistency and clarity. This step is crucial for accurately representing the causal relationships between variables in the DAGs.


In [ ]:
dataloaders = {}
original_observations_training = {} 
lagged_flattened_observations_training = {} 
flattened_dags_training = {} 

for error_dist in ['gaussian', 'uniform', 'laplace']:
    dataloader = DataLoader(n_variables = N_VARS,
                            maxlags = MAXLAGS)
    dataloader.from_pickle(f'data/training_data_{error_dist}.pkl')
    
    dataloaders[error_dist] = dataloader
    original_observations_training[error_dist] = dataloader.get_original_observations()
    lagged_flattened_observations_training[error_dist] = dataloader.get_observations()
    flattened_dags_training[error_dist] = dataloader.get_dags()


original_observations_list_training = []
for obs_list in original_observations_training.values():
    original_observations_list_training.extend(obs_list) 

lagged_flattened_observations_list_training = []
for obs_list in lagged_flattened_observations_training.values():
    lagged_flattened_observations_list_training.extend(obs_list)

flattened_dags_list_training = []
for dags_list in flattened_dags_training.values():
    flattened_dags_list_training.extend(dags_list)


We are now ready for the core of our methodology: the D2C method. <br>
This method starts from a list of observations and dags and computes the corresponding descritpors, storing them in a dataframe. <br>
The D2C class gets the following arguments: 
- `observations` (list): List of observations (pd.DataFrame) corresponding to each DAG.
- `dags` (list): List of directed acyclic graphs (DAGs) representing causal relationships.
- `couples_to_consider_per_dag` (int, optional): To speedup, one can consider only a limited number of possible couples of variables. Couples are chosen to respect a specific ratio of causal/noncausal. Therefore, a DAG must be available: it can only be used for training data. For testing purposes only, we recommend using `D2CWrapper` instead. Defaults to -1 (all couples).  
- `n_variables` (int, optional): Number of variables in the time series. Defaults to 3.
- `maxlags` (int, optional): Maximum number of lags in the time series. Defaults to 3.
- `seed` (int, optional): Random seed for reproducibility. Defaults to 42.
- `n_jobs` (int, optional): Number of parallel jobs to run. Defaults to 1.
 - `full` (bool, optional): computes the whole set of descriptors rather than just a subset. Defaults to True. 



In [ ]:
d2c_new = D2C(observations=lagged_flattened_observations_list_training,
        dags=flattened_dags_list_training, 
        couples_to_consider_per_dag=5, 
        n_variables=N_VARS, 
        maxlags=MAXLAGS,
        seed=42,
        n_jobs=30,
        full=True,
        dynamic=True,
        mb_estimator='ts',
        )

d2c_new.initialize()

In [ ]:
d2c_new.get_descriptors_df().to_pickle('data/descriptors_test_subset.pkl')

In [ ]:
dataloaders = {}
original_observations_testing = {} 
lagged_flattened_observations_testing = {} 
flattened_dags_testing = {}
true_causal_dfs = {}

for error_dist in ['gaussian', 'uniform', 'laplace']:
    dataloader = DataLoader(n_variables = N_VARS,
                            maxlags = MAXLAGS)
    dataloader.from_pickle(f'data/testing_data_{error_dist}.pkl')
    
    dataloaders[error_dist] = dataloader
    original_observations_testing[error_dist] = dataloader.get_original_observations()
    lagged_flattened_observations_testing[error_dist] = dataloader.get_observations()
    flattened_dags_testing[error_dist] = dataloader.get_dags()
    true_causal_dfs[error_dist] = dataloader.get_true_causal_dfs()

original_observations_list_testing = []
for obs_list in original_observations_testing.values():
    original_observations_list_testing.extend(obs_list) 

lagged_flattened_observations_list_testing = []
for obs_list in lagged_flattened_observations_testing.values():
    lagged_flattened_observations_list_testing.extend(obs_list)

flattened_dags_list_testing = []
for dags_list in flattened_dags_testing.values():
    flattened_dags_list_testing.extend(dags_list)

true_causal_dfs_list_testing = []
for causal_df in true_causal_dfs.values():
    true_causal_dfs_list_testing.extend(causal_df)

In [ ]:
from d2c.descriptors import D2C, DataLoader
d2c_new = D2C(observations=lagged_flattened_observations_list_testing,
        dags=flattened_dags_list_testing, 
        couples_to_consider_per_dag=5, 
        n_variables=N_VARS, 
        maxlags=MAXLAGS,
        seed=42,
        n_jobs=N_JOBS,
        full=True,
        dynamic=True,
        mb_estimator='ts',
        )

d2c_new.initialize()
d2c_new.get_descriptors_df().to_pickle('data/descriptors_test_subset.pkl')

We remind our variable naming convention: <br>
- A time series of `n_variables` dimensions will have names from 0 to `n_variables - 1` to refer to the variables at time `t` (present)
- names from `n_variables` to `n_variables*2 - 1` will indicate the same variable at time `t-1` (1-lag)
- names from `n_variables*2` to `n_variables*3 - 1` will indicate the same variable at time `t-2` (2-lag)  <br>
For example if `n_variables = 5`, the line where `edge_source` is 12 and `edge_dest` is 4, refers to the link between variable `3` at `t-2` to variable `5` at time `t`

In [5]:
from d2c.descriptors import D2C, DataLoader

dataloader = DataLoader(n_variables = 5,
                        maxlags = 3)
dataloader.from_pickle(f'data/observations/testing_data_gaussian.pkl')

In [2]:
dataloader

,graph_id,edge_source,edge_dest,is_causal,parcorr_errors,errors_correlation_with_inputs,coeff_cause,coeff_eff,HOC_3_1,HOC_1_2,...,mca_mca_cau_std,mbe_mbe_eff_interaction,mbe_mbe_eff_mean,mbe_mbe_eff_std,mca_mef_cau_interaction,mca_mef_eff_interaction,eff_m_cau_interaction,cau_eff_mbeff_plus_interaction,m_eff_interaction,mca_mca_cau_interaction
0,12,11,1,1,-0.267324,-0.061601,-0.277732,-0.271461,-0.483018,0.076301,...,0.000000,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,12,18,4,1,0.295563,0.056464,0.254020,0.314753,1.302380,0.216850,...,0.000000,0,0.0,0.0,0.0,0.000000,0.000000,0.065867,0.035894,0.0
2,12,17,1,1,0.248701,0.057119,0.240135,0.282859,0.577839,-0.101645,...,0.000000,0,0.0,0.0,0.0,0.000000,0.049506,0.000000,0.049506,0.0
3,12,12,1,1,-0.294143,-0.064142,-0.277217,-0.311544,-0.368090,-0.046006,...,0.000000,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,12,13,4,1,-0.138557,-0.023743,-0.074352,-0.148495,0.551401,0.054855,...,0.003179,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80995,1078,10,1,0,0.028771,-0.003351,0.007658,-0.015590,0.207283,0.011072,...,0.000000,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
80996,1078,13,0,0,-0.004670,-0.000077,-0.001852,-0.001850,0.160536,-0.251890,...,0.000000,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
80997,1078,6,0,0,-0.022817,0.000128,0.007615,0.000423,0.105431,-0.389524,...,0.000000,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
80998,1078,15,3,0,0.034719,-0.001925,-0.016797,-0.013226,-0.067560,0.008307,...,0.000000,0,0.0,0.0,0.0,0.038388,0.000000,0.000000,0.035801,0.0
